# Yardstick Heuristic Evaluation

This notebook mirrors `eval_heuristic.py` but is broken into small, explained steps.

We compute a simple heuristic score for each founder, summarize the splits, and then
evaluate ranking quality with NDCG by industry and founding year cohorts.


## Phase 1: Environment Setup

Set up paths so we can import the `heuristic` module from the yardstick benchmark,
and locate the project root dynamically based on the presence of `solution_sets`.


In [1]:
from __future__ import annotations

import math
import sys
from pathlib import Path

import pandas as pd


def _project_root() -> Path:
    """Locate the repo root (directory containing 'solution_sets')."""
    current = Path.cwd().resolve()
    for parent in (current, *current.parents):
        if (parent / "solution_sets").is_dir():
            return parent
    raise RuntimeError("Could not locate project root; please run from inside the gv_case_study repo.")


PROJECT_ROOT = _project_root()

YARDSTICK_ROOT = PROJECT_ROOT / "solution_sets" / "@ simple_benchmarks" / "yardstick"
if str(YARDSTICK_ROOT) not in sys.path:
    sys.path.append(str(YARDSTICK_ROOT))

XGB_ROOT = PROJECT_ROOT / "xgboost"
if str(XGB_ROOT) not in sys.path:
    sys.path.append(str(XGB_ROOT))

from heuristic import compute_heuristic_score
from src.models.metrics import calculate_mean_ndcg, calculate_mean_recall


## Phase 2: Load Post-Split Feature Tables

Load the precomputed founder-level feature tables for the train/val/test splits.
These CSVs are produced once by the yardstick pipeline and re-used here for evaluation.


In [2]:
def _datasets_root() -> Path:
    return PROJECT_ROOT / "solution_sets" / "@ datasets" / "processed_data"


def _read_split(name: str) -> pd.DataFrame:
    path = _datasets_root() / f"{name}_post_split_features.csv"
    df = pd.read_csv(path)
    print(f"Loaded {name} split from {path} with shape {df.shape}")
    return df


train = _read_split("train")
val = _read_split("val")
test = _read_split("test")


Loaded train split from /Users/yugu/Desktop/gehirn/gv_case_study/solution_sets/@ datasets/processed_data/train_post_split_features.csv with shape (3335, 20)
Loaded val split from /Users/yugu/Desktop/gehirn/gv_case_study/solution_sets/@ datasets/processed_data/val_post_split_features.csv with shape (714, 20)
Loaded test split from /Users/yugu/Desktop/gehirn/gv_case_study/solution_sets/@ datasets/processed_data/test_post_split_features.csv with shape (717, 20)


## Phase 3: Compute Heuristic Scores and Summaries

Apply `compute_heuristic_score` to each split to get a simple ranking score per founder,
merge it back onto the original feature frames, and print basic summary statistics.


In [3]:
def _compute_scores(frame: pd.DataFrame, use_company: bool) -> pd.DataFrame:
    scored = compute_heuristic_score(frame, use_company=use_company)
    out = frame.merge(scored[["person_id", "heuristic_score"]], on="person_id", how="left")
    return out


def _print_summary(df: pd.DataFrame, name: str) -> None:
    print(f"[{name}] rows={len(df)} cols={len(df.columns)} score_mean={df['heuristic_score'].mean():.4f}")


scored_train = _compute_scores(train, use_company=False)
scored_val = _compute_scores(val, use_company=False)
scored_test = _compute_scores(test, use_company=False)

_print_summary(scored_train, "train")
_print_summary(scored_val, "val")
_print_summary(scored_test, "test")


[train] rows=3335 cols=21 score_mean=0.4085
[val] rows=714 cols=21 score_mean=0.4239
[test] rows=717 cols=21 score_mean=0.3939


## Phase 4: Evaluate Ranking Quality with NDCG and Recall

Use Normalized Discounted Cumulative Gain (NDCG) and Recall at k to evaluate how well the heuristic 
ordering of founders aligns with ground-truth relevance labels. We compute per-industry metrics and 
their macro averages across industries using the shared `calculate_mean_ndcg` and `calculate_mean_recall` helpers.


In [4]:
# Group founders by industry when building groups for mean metrics.
def _build_group_arrays(
    frame: pd.DataFrame,
    label_column: str,
    score_column: str,
    cohort_columns=("industry",),
    min_cohort_size: int = 2,
):
    """Flatten industry groups into labels/scores/groups for mean metrics."""
    all_labels: list[float] = []
    all_scores: list[float] = []
    groups: list[int] = []
    for _, grp in frame.groupby(list(cohort_columns)):
        if len(grp) < min_cohort_size:
            continue
        labels = grp[label_column].tolist()
        scores = grp[score_column].tolist()
        all_labels.extend(labels)
        all_scores.extend(scores)
        groups.append(len(labels))
    return all_labels, all_scores, groups


for name, df in ("train", scored_train), ("val", scored_val), ("test", scored_test):
    # Macro metrics across industries
    labels, scores, groups = _build_group_arrays(
        df, label_column="label", score_column="heuristic_score", cohort_columns=("industry",),
    )
    ndcg_macro = calculate_mean_ndcg(labels=labels, scores=scores, groups=groups, eval_metric="ndcg@20")
    recall_macro = calculate_mean_recall(labels=labels, scores=scores, groups=groups, eval_metric="recall@20")
    print(f"[{name}] macro_mean_ndcg@20={ndcg_macro:.4f} macro_mean_recall@20={recall_macro:.4f}")

    # Per-industry metrics using the same helpers (one group per industry)
    for industry, grp in df.groupby("industry"):
        if len(grp) < 2:
            continue
        ind_labels = grp["label"].tolist()
        ind_scores = grp["heuristic_score"].tolist()
        ind_groups = [len(ind_labels)]
        ind_ndcg = calculate_mean_ndcg(labels=ind_labels, scores=ind_scores, groups=ind_groups, eval_metric="ndcg@20")
        ind_recall = calculate_mean_recall(labels=ind_labels, scores=ind_scores, groups=ind_groups, eval_metric="recall@20")
        print(f"\t[{name}][{industry}] ndcg@20={ind_ndcg:.4f}   recall@20={ind_recall:.4f}")


[train] macro_mean_ndcg@20=0.2563 macro_mean_recall@20=0.2016
	[train][Healthcare and Biotech] ndcg@20=0.2522   recall@20=0.3043
	[train][Information Technology] ndcg@20=0.1361   recall@20=0.0102
	[train][Other] ndcg@20=0.3807   recall@20=0.2903
[val] macro_mean_ndcg@20=0.5334 macro_mean_recall@20=0.3889
	[val][Healthcare and Biotech] ndcg@20=0.2910   recall@20=0.0000
	[val][Information Technology] ndcg@20=0.5612   recall@20=0.5000
	[val][Other] ndcg@20=0.7482   recall@20=0.6667
[test] macro_mean_ndcg@20=0.3600 macro_mean_recall@20=0.3333
	[test][Healthcare and Biotech] ndcg@20=0.5506   recall@20=0.0000
	[test][Information Technology] ndcg@20=0.5293   recall@20=1.0000
	[test][Other] ndcg@20=0.0000   recall@20=0.0000
